In [1]:
import pandas as pd
import joblib  
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("clean_restaurants.csv")

In [3]:
df["Name"] = df["Name"].fillna("")
df["Location"] = df["Location"].fillna("")
df["Locality"] = df["Locality"].fillna("")
df["City"] = df["City"].fillna("")
df["Cuisine"] = df["Cuisine"].fillna("")
df["Rating"] = df["Rating"].fillna(0)
df["Votes"] = df["Votes"].fillna(0)
df["Cost"] = df["Cost"].fillna(0)

In [4]:
df["Features"] = (
    df["City"].astype(str) + " " +
    df["Locality"].astype(str) + " " +
    df["Cuisine"].astype(str))

In [5]:
tfidf = TfidfVectorizer(stop_words="english")

feature_matrix = tfidf.fit_transform(df["Features"])

In [6]:
similarity = cosine_similarity(feature_matrix)

In [7]:
joblib.dump(tfidf, "restaurant_vectorizer.pkl")
joblib.dump(similarity, "restaurant_similarity.pkl")

print("Restaurant Recommendation Model Saved Successfully!")

Restaurant Recommendation Model Saved Successfully!


In [8]:
def recommend_restaurants(city, budget, top_n=5):

    # Filter by City
    filtered = df[df["City"].str.lower() == city.lower()]

    # Filter by Budget
    filtered = filtered[filtered["Cost"] <= budget]

    # Check if restaurants are found
    if filtered.empty:
        return "No Restaurants Found"

    # Sort by Rating and Votes
    filtered = filtered.sort_values(
        by=["Rating", "Votes"],
        ascending=False
    )

    # Return Top Restaurants
    return filtered[
        [
            "Name",
            "Rating",
            "Votes",
            "Cost",
            "Location",
            "City"
        ]
    ].head(top_n)

In [9]:
result = recommend_restaurants(
    city="mumbai",
    budget=600
)

print(result)

                   Name  Rating  Votes  Cost  \
1749        Shake Me Up     5.0      2   500   
1155      Pizza Caprina     5.0      1   400   
1202      Rome Pizzeria     5.0      1   400   
1208          Wowfillss     5.0      1   500   
1248  The Pancake Story     5.0      1   500   

                                          Location    City  
1749                      Mira Road, North Western  Mumbai  
1155  Premier Emperor Building,Marol, Andheri East  Mumbai  
1202       Near Andheri East Station, Andheri East  Mumbai  
1208                           Parel, South Mumbai  Mumbai  
1248   Narayan Niwas,Vile Parle East, Andheri East  Mumbai  
